# <font color="Green">**Notebook Purpose**</font>

This notebook constructs two complementary representations of each patient's longitudinal medication trajectory using the `medication_info` table produced in earlier preprocessing steps. Both representations encode which medication classes a patient was prescribed in each half-year period between 2019 and 2024, enabling downstream analyses of temporal treatment patterns (e.g., clustering, trajectory similarity, or embedding methods).

* **`patient_bins`**: A dictionary where keys are `patient_id`s and values are lists of 12 sets. Each position in the list corresponds to a half-year bin (Jan–Jun or Jul–Dec for 2019–2024), and each set contains the medication classes prescribed during that interval. This representation preserves temporal ordering and allows for flexible, set-based operations on treatment histories.

* **`patient_vectors`**: A dictionary where keys are `patient_id`s and values are fixed-length vectors created by one-hot encoding the medication classes within each half-year bin and concatenating these encodings across all 12 bins. This yields a compact trajectory vector for each patient that is suitable for use in distance-based methods, clustering, or machine learning models.

---

### <font color="Red">Required Data</font>

To run this notebook, you will need the following input files, already generated by the earlier preprocessing notebook (e.g., `MainTableCreation.ipynb`) and stored in the working directory (or an equivalent path update in the code):

1. **`patient_demographics.csv`**  
   Although not directly transformed in this notebook, this file is loaded to maintain consistency with the overall pipeline and to allow for potential cross-checks. It should contain at least:
   * `patient_id`
   * Basic demographic fields (e.g., age, sex, race/ethnicity) as defined in the preprocessing pipeline.

2. **`medication_info.csv`**  
   This is the primary input to the notebook. It must contain one row per patient–medication–start-date record, with at least the following columns:
   * `patient_id` – unique patient identifier.
   * `medication_class` – medication class label.
   * `start_date` – prescription start date, used to assign each record to one of 12 half-year bins between 2019 and 2024.


With these inputs available, the notebook will output the `patient_bins` and `patient_vectors` Python objects in memory and (optionally) export them for use in subsequent analysis notebooks (e.g., clustering or trajectory modeling).


In [ ]:
import pandas as pd
import numpy as np
import pickle

In [ ]:
patient_demographics = pd.read_csv('/content/patient_demographics.csv')
medication_info = pd.read_csv('/content/medication_info.csv')

###<font color="black">**Creating patient_bins**</font>

In [ ]:
# This may look strange, but I am just ensuring that any date between 2019-2024 gets mapped to an index
# between 0-11. With 0 being Jan-Jun 2019 and 11 being Jul-Dec 2024.
def get_half_year_index(date):
    return (date.year - 2019) * 2 + (0 if date.month <= 6 else 1)

In [ ]:
medication_info['start_date'] = pd.to_datetime(medication_info['start_date'])

patient_bins = {}

# Iterating through each patients info. Can think of it as creating a mini data frame for each patient.
for patient_id, patient_info in medication_info.groupby('patient_id'):

    # This sorts the patients medication history by date.
    patient_info = patient_info.sort_values(by='start_date')

    for _, row in patient_info.iterrows():

        medication_class = row['medication_class']

        start_date = row['start_date']

        half_year_index = get_half_year_index(start_date)


        if 0 <= half_year_index < 12:

          if patient_id not in patient_bins:
              patient_bins[patient_id] = [set() for _ in range(12)]

          patient_bins[patient_id][half_year_index].add(medication_class)

        else:
            print(f"Invalid half-year index: {half_year_index}")

Now I will impute 'nothing' into the emtpy sets.

In [ ]:
for patient_id, bins in patient_bins.items():
    for i, bin in enumerate(bins):
        if not bin:
            bins[i] = {'nothing'}

Example patient from patient_bins

In [ ]:
unique_ids = medication_info['patient_id'].unique()

random_patient_id = np.random.choice(unique_ids)

print('Patient id: ', random_patient_id)

patient_bins[random_patient_id]

###<font color="black">**Creating patient_vectors**</font>

**Approach:** will use a one-hot encoding approach to create these vectors. Each bin, which represents a half year period of medication data, will be represented by a 9 dimensional vector (one dimension for each medication class). That means that each patient will then be assigned 12 vectors (one for each half year period). Then, starting from the vector that represents 2019H1, each vector will be prepended onto the end of the previous one, creating a 108 dimensional vector.

**Reasoning:** This approach achieves two things. The first thing it does is translate our data into a vector, allowing it to exist in a euclidean space (easier for clustering). The other thing it does is preserve the temporal nature of our data. By prepending the vectors for each bin in order of relevance, we ensure that patients who took similar medications at similar times will be near eachother in euclidean space.

**Here are the indices in the vectors for each medication class**

**MET:** [1, 0, 0, 0, 0, 0, 0, 0, 0]

**SUL:** [0, 1, 0, 0, 0, 0, 0, 0, 0]

**SGLT2:** [0, 0, 1, 0, 0, 0, 0, 0, 0]

**GLP-1:** [0, 0, 0, 1, 0, 0, 0, 0, 0]

**GIP/GLP-1:** [0, 0, 0, 0, 1, 0, 0, 0, 0]

**Insulin:** [0, 0, 0, 0, 0, 1, 0, 0, 0]

**DPP-4:** [0, 0, 0, 0, 0, 0, 1, 0, 0]

**TZD:** [0, 0, 0, 0, 0, 0, 0, 1, 0]

**Other:** [0, 0, 0, 0, 0, 0, 0, 0, 1]

In [ ]:
med_class_to_index = {
    'MET': 0,
    'SUL': 1,
    'SGLT2': 2,
    'GLP-1': 3,
    'GIP/GLP-1': 4,
    'Insulin': 5,
    'DPP-4': 6,
    'TZD': 7,
    'Other': 8
}

patient_vectors = {}

for patient_id, bins in patient_bins.items():
    trajectory_vectors = []

    for bin_set in bins:
        vector = [0] * 10
        for med_class in bin_set:
            if med_class in med_class_to_index:
                vector[med_class_to_index[med_class]] = 1
            else:
                print(f"Unknown medication class: {med_class}")
        trajectory_vectors.append(vector)

    # Flatten the 12 x 10 matrix into a 108-dimensional vector
    patient_vectors[patient_id] = np.array(trajectory_vectors).flatten()

Now lets take a look at the resulting representation and compare it to the representation held in patient_bins.

In [ ]:
random_patient_id = np.random.choice(unique_ids)

print('Patient id: ', random_patient_id)

print(patient_bins[random_patient_id])
print(patient_vectors[random_patient_id])

###<font color="black">**Exporting the data structures**</font>


In [ ]:
with open("patient_bins.pkl", "wb") as f:
    pickle.dump(patient_bins, f)

with open("patient_vectors.pkl", "wb") as f:
    pickle.dump(patient_vectors, f)